# H1N1 Hackathon – Dataset B (Colab)
Run all cells top to bottom. When prompted, upload `dataset_B_training.csv` and `dataset_B_testing.csv`. Submission CSVs download automatically at the end.

Decision rule: send a vaccine only if `P(accept)·(Hu−Hv) > 500·(1−P)`; models are compared by **total town cost**.

In [1]:
!pip -q install lightgbm

In [2]:
import os
from google.colab import files
if not (os.path.exists('/content/drive/MyDrive/dataset_B_training.csv') and os.path.exists('/content/drive/MyDrive/dataset_B_testing.csv')):
    files.upload()   # select both CSVs

Saving dataset_B_testing.csv to dataset_B_testing.csv
Saving dataset_B_training.csv to dataset_B_training.csv


In [3]:
import warnings
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False                     # pip install lightgbm to enable

SEED = 42
SCENARIO = 2          # 2 = person not vaccinated in time if we fail to send one (see notes)
VACCINE_COST, HOSP_COST = 500, 25_000

# Hospitalisation-risk assumptions (adjust to your consensus / report)
BASE_RISK  = 0.0015   # P(hosp | infected, unvaccinated, age 35-44, no chronic conditions)
AGE_MULT   = {"18 - 34 Years": 0.8, "35 - 44 Years": 1.0, "45 - 54 Years": 1.2,
              "55 - 64 Years": 1.5, "65+ Years": 1.8}
CHRONIC_K  = 2.0      # each chronic condition adds 200 % relative risk
VAX_EFFECT = 0.60     # vaccine effectiveness against hospitalisation

## 1. Load

In [4]:
train = pd.read_csv("dataset_B_training.csv")
test = pd.read_csv("dataset_B_testing.csv")
y = train.pop("h1n1_vaccine").values
ids = test["respondent_id"].copy()
train = train.drop(columns="respondent_id")
test = test.drop(columns="respondent_id")
n = len(train)
print("train", train.shape, "test", test.shape, "| vaccinated share %.3f" % y.mean())

train (4756, 29) test (4749, 29) | vaccinated share 0.394


## 2. Pre-processing (two views: one-hot for linear/sklearn, native categoricals for LightGBM)

In [5]:
CAT = [c for c in train.columns if train[c].dtype == object or str(train[c].dtype) in ("str", "string")]
ORD = {"age_group": list(AGE_MULT),
       "education": ["< 12 Years", "12 Years", "Some College", "College Graduate"]}
MISS_COLS = ["health_insurance", "doctor_recc_h1n1", "employment_sector",
             "income_poverty", "education"]


def prep(df, kind):
    d = df.copy()
    for c in MISS_COLS:                                   # missingness is informative
        d[c + "_miss"] = d[c].isna().astype(int)
    for c, order in ORD.items():
        d[c + "_ord"] = d[c].map({v: i for i, v in enumerate(order)})
    beh = [c for c in d if c.startswith("behavioral")]
    d["n_precautions"] = d[beh].sum(axis=1)
    d["opinion_gap"] = d["opinion_h1n1_vacc_effective"] - d["opinion_h1n1_sick_from_vacc"]
    if kind == "ohe":
        d = pd.get_dummies(d, columns=CAT, dtype=float)
        d = d.fillna(d.median())
    else:
        for c in CAT:
            d[c] = d[c].fillna("MISSING").astype("category")
    return d


full = pd.concat([train, test])                           # concat so dummy columns align
X_ohe, X_tree = prep(full, "ohe"), prep(full, "tree")
views = {"ohe": (X_ohe.iloc[:n], X_ohe.iloc[n:]),
         "tree": (X_tree.iloc[:n], X_tree.iloc[n:])}

## 3. Cost model

In [6]:
def hosp_probs(df, base=BASE_RISK, ve=VAX_EFFECT, k=CHRONIC_K):
    """(P(hosp | unvaccinated), P(hosp | vaccinated)) per person."""
    m = df["age_group"].map(AGE_MULT).values * (1 + k * df["number_chronic_med_conditions"].values)
    p_u = np.clip(base * m, 0, 1)
    return p_u, p_u * (1 - ve)


def total_cost(y_true, pred, Hu, Hv, scenario=SCENARIO):
    """Vaccines supplied or reimbursed + expected hospitalisation cost."""
    y_true, pred = np.asarray(y_true), np.asarray(pred)
    vaccine = VACCINE_COST * ((pred == 1) | (y_true == 1))
    protected = (y_true == 1) & ((pred == 1) | (scenario == 1))
    return vaccine.sum() + np.where(protected, Hv, Hu).sum()


def decide(proba, Hu, Hv, scenario=SCENARIO):
    """Send a vaccine iff  P * (Hu - Hv) > 500 * (1 - P)."""
    if scenario == 1:
        return np.zeros(len(proba), int)                  # sending never helps in scenario 1
    return (proba * (Hu - Hv) > VACCINE_COST * (1 - proba)).astype(int)


p_u, p_v = hosp_probs(train)
Hu, Hv = HOSP_COST * p_u, HOSP_COST * p_v
base0 = total_cost(y, np.zeros(n, int), Hu, Hv)
base1 = total_cost(y, np.ones(n, int), Hu, Hv)
oracle = total_cost(y, y, Hu, Hv)
print(f"Cost per person -> all-0: ${base0/n:,.2f} | all-1: ${base1/n:,.2f} | oracle: ${oracle/n:,.2f}")

Cost per person -> all-0: $288.73 | all-1: $567.52 | oracle: $264.53


## 4. Models + 5-fold out-of-fold probabilities

In [7]:
models = {
    "LogReg": (make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=2000)), "ohe"),
    "HistGB": (HistGradientBoostingClassifier(learning_rate=0.05, max_leaf_nodes=8, min_samples_leaf=50,
                                              l2_regularization=1, max_iter=200, random_state=SEED), "ohe"),
}
models["RandomForest"] = (RandomForestClassifier(n_estimators=400, min_samples_leaf=10, max_features=0.3,
                                                  n_jobs=-1, random_state=SEED), "ohe")
if HAS_LGB:
    models["LightGBM"] = (lgb.LGBMClassifier(learning_rate=0.05, num_leaves=8, n_estimators=300,
                                             min_child_samples=50, colsample_bytree=0.8, reg_lambda=1,
                                             random_state=SEED, verbose=-1), "tree")

skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
folds = list(skf.split(np.zeros(n), y))
oof, test_proba = {}, {}
for name, (est, kind) in models.items():
    Xa, Xb = views[kind]
    o = np.zeros(n)
    for a, b in folds:
        o[b] = clone(est).fit(Xa.iloc[a], y[a]).predict_proba(Xa.iloc[b])[:, 1]
    oof[name] = o
    test_proba[name] = clone(est).fit(Xa, y).predict_proba(Xb)[:, 1]
    print(f"{name:9s} AUC={roc_auc_score(y, o):.4f}  logloss={log_loss(y, o):.4f}  brier={brier_score_loss(y, o):.4f}")

oof["Blend"] = np.mean([oof[k] for k in models], axis=0)
test_proba["Blend"] = np.mean([test_proba[k] for k in models], axis=0)

LogReg    AUC=0.8490  logloss=0.4656  brier=0.1503
HistGB    AUC=0.8517  logloss=0.4603  brier=0.1486
RandomForest AUC=0.8509  logloss=0.4658  brier=0.1503
LightGBM  AUC=0.8524  logloss=0.4591  brier=0.1483


## 5. Cost-based evaluation

In [8]:
rows = []
for k, pr in oof.items():
    pred = decide(pr, Hu, Hv)
    tot = total_cost(y, pred, Hu, Hv)
    fold_costs = [total_cost(y[b], pred[b], Hu[b], Hv[b]) for _, b in folds]
    rows.append(dict(model=k, total_cost=tot, cost_per_person=tot / n,
                     saving_vs_all0=base0 - tot,
                     pct_of_oracle_saving=100 * (base0 - tot) / (base0 - oracle),
                     fold_sd=np.std(fold_costs), share_sent=pred.mean()))
results = pd.DataFrame(rows).set_index("model").sort_values("total_cost")
print("\n", results.round(2).to_string())


               total_cost  cost_per_person  saving_vs_all0  pct_of_oracle_saving  fold_sd  share_sent
model                                                                                               
LightGBM      1347880.25           283.41        25323.50                 22.00  2738.61        0.09
LogReg        1347939.00           283.42        25264.75                 21.95  2762.56        0.08
HistGB        1348464.50           283.53        24739.25                 21.50  2934.92        0.07
Blend         1348742.00           283.59        24461.75                 21.26  3180.89        0.07
RandomForest  1349019.75           283.65        24184.00                 21.01  2741.37        0.06


## 6. Submissions (decision = send vaccine)

In [9]:
pu_te, pv_te = hosp_probs(test)
for k in results.index:
    pred_te = decide(test_proba[k], HOSP_COST * pu_te, HOSP_COST * pv_te)
    pd.DataFrame({"respondent_id": ids, "h1n1_vaccine": pred_te}).to_csv(f"submission_{k}.csv", index=False)
    print(f"submission_{k}.csv  -> share predicted 1 on test: {pred_te.mean():.2%}")


submission_LightGBM.csv  -> share predicted 1 on test: 8.87%
submission_LogReg.csv  -> share predicted 1 on test: 8.49%
submission_HistGB.csv  -> share predicted 1 on test: 7.66%
submission_Blend.csv  -> share predicted 1 on test: 7.22%
submission_RandomForest.csv  -> share predicted 1 on test: 6.42%


## 7. Download submissions

In [10]:
import glob
for f in sorted(glob.glob('submission_*.csv')):
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>